# Build MCP Servers

In this notebook, we explore and understand the MCP (Model Context Protocol) servers that form the Tool Layer of our research harness.

## Learning Objectives
- Understand how MCP servers expose tools as standardized services
- Examine the Document, Search, and Analysis MCP server implementations
- Learn how the harness controller connects to MCP tools

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

print("Environment loaded successfully")

## 1. Document MCP Server Structure

The Document MCP server wraps Docling functionality into three tools:
- `ingest_document` — Parse, chunk, embed, and store documents
- `get_document_status` — Check processing status
- `list_documents` — List all ingested documents

In [ ]:
# Examine the Document MCP server tool definitions
from mcp_servers.doc_mcp.server import list_tools as doc_list_tools
import asyncio

doc_tools = asyncio.get_event_loop().run_until_complete(doc_list_tools())
for tool in doc_tools:
    print(f"\n🔧 {tool.name}")
    print(f"   Description: {tool.description}")
    print(f"   Parameters: {list(tool.inputSchema.get('properties', {}).keys())}")

## 2. Search MCP Server Structure

The Search MCP server provides semantic search capabilities:
- `semantic_search` — Find similar chunks across all documents
- `search_by_document` — Search within a specific document
- `get_chunk_context` — Get surrounding chunks for context

In [ ]:
from mcp_servers.search_mcp.server import list_tools as search_list_tools

search_tools = asyncio.get_event_loop().run_until_complete(search_list_tools())
for tool in search_tools:
    print(f"\n🔧 {tool.name}")
    print(f"   Description: {tool.description}")
    print(f"   Parameters: {list(tool.inputSchema.get('properties', {}).keys())}")

## 3. Analysis MCP Server Structure

The Analysis MCP server provides LLM-powered capabilities:
- `rewrite_query` — Generate optimized sub-queries
- `synthesize_context` — Combine passages into coherent summaries
- `generate_research_plan` — Create structured research plans
- `draft_report` — Write research reports from context

In [ ]:
from mcp_servers.analysis_mcp.server import list_tools as analysis_list_tools

analysis_tools = asyncio.get_event_loop().run_until_complete(analysis_list_tools())
for tool in analysis_tools:
    print(f"\n🔧 {tool.name}")
    print(f"   Description: {tool.description}")
    print(f"   Required: {tool.inputSchema.get('required', [])}")

## 4. MCP Server Deployment Pattern

Each MCP server can run in two modes:
1. **stdio mode** — for local development (stdin/stdout communication)
2. **HTTP mode** — for production deployment (REST API)

The harness controller connects to them via the configured URLs in `.env`:
```
DOC_MCP_URL=http://localhost:9001
SEARCH_MCP_URL=http://localhost:9002
ANALYSIS_MCP_URL=http://localhost:9003
```

In [ ]:
# In this lab, we use direct function calls (equivalent to stdio mode)
# The tool layer in the orchestrator uses these same functions
from agents.orchestrator.layers.tools import (
    semantic_search,
    rewrite_query,
    synthesize_context,
    generate_plan,
    draft_report,
)

print("Tool layer functions available:")
print("  - semantic_search(query, top_k)")
print("  - rewrite_query(query)")
print("  - synthesize_context(query, passages)")
print("  - generate_plan(query, iteration, failure_hints, context)")
print("  - draft_report(query, context, plan, previous_draft, hints)")

## Summary

The Tool Layer provides:
- **Standardized interfaces** via MCP protocol
- **Independent deployability** — each server scales separately
- **Testability** — each tool can be verified in isolation

Next: Test each MCP server's tools independently →